# Pseudocode to Code: IIR Low-Pass Filter on IQ Signals

This notebook follows the **pseudocode** skill conventions (`.agents/skills/pseudocode/SKILL.md`) to formally design an IIR low-pass filter algorithm, and then translates it directly into executable Python/NumPy/SciPy code.

---

### Objectives
1. Formulate formal, language-agnostic pseudocode for synthetic IQ generation, filter design, filtering, and power computation.
2. Map the pseudocode constructs 1:1 to Python (`numpy` and `scipy.signal`).
3. Verify that input/output shapes adhere to canonical `(N, 2, L)`.
4. Validate signal power attenuation after filtering ($P_{\text{after}} < P_{\text{before}}$).

## 1. Formal Pseudocode Specification

Adhering to the **Hard Rules** from the pseudocode skill:
- Assignment: `←`
- Comparison / Equality: `=`
- Explicit block closings: `END FUNCTION`
- PascalCase for algorithms, camelCase for variables
- Explicit type declarations

```text
ALGORITHM IirFilterIqWorkflow

FUNCTION GenerateSyntheticIq(numExamples, numSamples, seed) → ARRAY
    DECLARE rng AS RandomGenerator
    DECLARE timeVec AS ARRAY
    DECLARE baseI, baseQ AS ARRAY
    DECLARE noiseI, noiseQ AS ARRAY
    DECLARE I, Q AS ARRAY
    DECLARE datasetX AS ARRAY

    rng ← InitializeRng(seed)
    timeVec ← LinearSpaced(0, 1, numSamples, endpoint=FALSE)
    baseI ← Cosine(2 * PI * 5.0 * timeVec)
    baseQ ← Sine(2 * PI * 5.0 * timeVec)
    
    noiseI ← StandardNormal(rng, shape=(numExamples, numSamples))
    noiseQ ← StandardNormal(rng, shape=(numExamples, numSamples))
    
    I ← Repeat(baseI, numExamples) + 0.6 * noiseI
    Q ← Repeat(baseQ, numExamples) + 0.6 * noiseQ
    
    datasetX ← Stack([I, Q], axis=1) // shape: (numExamples, 2, numSamples)
    RETURN datasetX
END FUNCTION

FUNCTION DesignButterworthIir(order, normalizedCutoff) → TUPLE
    DECLARE bCoeffs, aCoeffs AS ARRAY
    (bCoeffs, aCoeffs) ← ComputeButterworthCoefficients(order, normalizedCutoff, type="lowpass")
    RETURN (bCoeffs, aCoeffs)
END FUNCTION

FUNCTION ApplyIirFilter(signalArray, bCoeffs, aCoeffs, timeAxis) → ARRAY
    DECLARE filteredSignal AS ARRAY
    // Filter along axis 2 (time dimension) for each component
    filteredSignal ← FilterLinear(bCoeffs, aCoeffs, signalArray, axis=timeAxis)
    RETURN filteredSignal
END FUNCTION

FUNCTION ComputeIqPower(signalArray) → FLOAT
    DECLARE I, Q AS ARRAY
    DECLARE totalPower AS FLOAT
    I ← signalArray[:, 0, :]
    Q ← signalArray[:, 1, :]
    totalPower ← Mean(I^2 + Q^2)
    RETURN totalPower
END FUNCTION

// Main Orchestration
DECLARE X, XFiltered AS ARRAY
DECLARE b, a AS ARRAY
DECLARE powerBefore, powerAfter AS FLOAT

X ← GenerateSyntheticIq(numExamples=5, numSamples=1000, seed=42)
(b, a) ← DesignButterworthIir(order=2, normalizedCutoff=0.1)
XFiltered ← ApplyIirFilter(X, b, a, timeAxis=2)
powerBefore ← ComputeIqPower(X)
powerAfter ← ComputeIqPower(XFiltered)
ASSERT powerAfter < powerBefore
END ALGORITHM
```

## 2. Translation: Python / NumPy / SciPy Implementation

We translate the pseudocode functions 1:1 into Python, preserving identical structure, naming, and invariants.

In [1]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
def generate_synthetic_iq(num_examples: int = 5, num_samples: int = 1000, seed: int = 42) -> np.ndarray:
    """Translate GenerateSyntheticIq pseudocode."""
    rng = np.random.default_rng(seed)
    time_vec = np.linspace(0, 1, num_samples, endpoint=False, dtype=np.float32)
    
    base_i = np.cos(2 * np.pi * 5.0 * time_vec, dtype=np.float32)
    base_q = np.sin(2 * np.pi * 5.0 * time_vec, dtype=np.float32)
    
    noise_i = rng.standard_normal((num_examples, num_samples)).astype(np.float32)
    noise_q = rng.standard_normal((num_examples, num_samples)).astype(np.float32)
    
    i_comp = np.repeat(base_i[np.newaxis, :], num_examples, axis=0) + 0.6 * noise_i
    q_comp = np.repeat(base_q[np.newaxis, :], num_examples, axis=0) + 0.6 * noise_q
    
    dataset_x = np.stack([i_comp, q_comp], axis=1)  # canonical shape: (N, 2, L)
    return dataset_x

def design_butterworth_iir(order: int = 2, normalized_cutoff: float = 0.1) -> tuple[np.ndarray, np.ndarray]:
    """Translate DesignButterworthIir pseudocode using scipy.signal.butter."""
    # scipy.signal.butter: Wn is normalized to Nyquist frequency (1.0 = sample_rate / 2)
    b_coeffs, a_coeffs = signal.butter(N=order, Wn=normalized_cutoff, btype="lowpass")
    return b_coeffs.astype(np.float32), a_coeffs.astype(np.float32)

def apply_iir_filter(signal_array: np.ndarray, b_coeffs: np.ndarray, a_coeffs: np.ndarray, time_axis: int = 2) -> np.ndarray:
    """Translate ApplyIirFilter pseudocode using scipy.signal.lfilter."""
    return signal.lfilter(b_coeffs, a_coeffs, signal_array, axis=time_axis).astype(np.float32)

def compute_iq_power(signal_array: np.ndarray) -> float:
    """Translate ComputeIqPower pseudocode."""
    i_comp = signal_array[:, 0, :]
    q_comp = signal_array[:, 1, :]
    return float(np.mean(i_comp**2 + q_comp**2))

print("Pseudocode translated into Python functions.")

Pseudocode translated into Python functions.


## 3. Pipeline Execution and Verification

In [3]:
# Main execution flow
X = generate_synthetic_iq(num_examples=5, num_samples=1000, seed=42)
b, a = design_butterworth_iir(order=2, normalized_cutoff=0.1)
X_filtered = apply_iir_filter(X, b, a, time_axis=2)

# Compute power before and after
p_before = compute_iq_power(X)
p_after = compute_iq_power(X_filtered)
p_ratio = p_after / p_before

print("--- Execution Results ---")
print(f"X shape:          {X.shape} (dtype: {X.dtype})")
print(f"X_filtered shape: {X_filtered.shape} (dtype: {X_filtered.dtype})")
print(f"Filter b coeffs:  {b}")
print(f"Filter a coeffs:  {a}")
print(f"Power before:     {p_before:.5f}")
print(f"Power after:      {p_after:.5f}")
print(f"Power ratio:      {p_ratio:.5f}")

# Assertions
assert X.shape == (5, 2, 1000), f"Unexpected shape: {X.shape}"
assert X_filtered.shape == (5, 2, 1000), f"Unexpected filtered shape: {X_filtered.shape}"
assert p_after < p_before, "Power was not attenuated after low-pass filtering!"
print("\nAll assertions PASSED successfully!")

--- Execution Results ---
X shape:          (5, 2, 1000) (dtype: float32)
X_filtered shape: (5, 2, 1000) (dtype: float32)
Filter b coeffs:  [0.02008337 0.04016673 0.02008337]
Filter a coeffs:  [ 1.        -1.5610181  0.6413515]
Power before:     1.74036
Power after:      1.08406
Power ratio:      0.62289
All assertions PASSED successfully!


## 4. Visual Inspection: Signal and Constellation

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Time domain: I component
axes[0].plot(X[0, 0, :150], label="Raw I (Noisy)", color="cornflowerblue", alpha=0.8)
axes[0].plot(X_filtered[0, 0, :150], label="Filtered I (Butterworth)", color="navy", linewidth=2)
axes[0].set_title("In-phase (I) Signal (First 150 Samples)")
axes[0].set_xlabel("Sample Index")
axes[0].set_ylabel("Amplitude")
axes[0].legend()
axes[0].grid(True, linestyle="--", alpha=0.6)

# Constellation diagram
axes[1].scatter(X[0, 0, :], X[0, 1, :], s=12, alpha=0.4, color="salmon", label="Raw IQ")
axes[1].scatter(X_filtered[0, 0, :], X_filtered[0, 1, :], s=12, alpha=0.6, color="seagreen", label="Filtered IQ")
axes[1].set_title("IQ Constellation Diagram")
axes[1].set_xlabel("In-phase (I)")
axes[1].set_ylabel("Quadrature (Q)")
axes[1].legend()
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].axis("equal")

plt.tight_layout()
plt.show()